In [2]:
# Cell 1 — Import Libraries
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Libraries loaded!")
print(f"🖥️  Using device: {device}")

✅ Libraries loaded!
🖥️  Using device: cpu


In [1]:
print("hello")

hello


In [3]:
# Cell 2 — Load Processed Data
import pandas as pd

train = pd.read_csv("../data/processed/train_clean.csv")
valid = pd.read_csv("../data/processed/valid_clean.csv")
test  = pd.read_csv("../data/processed/test_clean.csv")

print("✅ Data loaded!")
print(f"Train: {len(train)} samples")
print(f"Valid: {len(valid)} samples")
print(f"Test:  {len(test)} samples")

print("\n=== Sample ===")
print(train.head(3))

✅ Data loaded!
Train: 10240 samples
Valid: 1284 samples
Test:  1267 samples

=== Sample ===
           id                                    clean_statement  \
0   2635.json  says the annies list political group supports ...   
1  10540.json  when did the decline of coal start it started ...   
2    324.json  hillary clinton agrees with john mccain by vot...   

  simplified_label  
0       Misleading  
1        Uncertain  
2          Factual  


In [4]:
# Cell 3 — Encode Labels
from sklearn.preprocessing import LabelEncoder

# Define label mapping
label2id = {
    "Factual":    0,
    "Misleading": 1,
    "Uncertain":  2
}

id2label = {v: k for k, v in label2id.items()}

# Apply encoding
train["label_id"] = train["simplified_label"].map(label2id)
valid["label_id"] = valid["simplified_label"].map(label2id)
test["label_id"]  = test["simplified_label"].map(label2id)

print("✅ Labels encoded!")
print("\n=== Label Mapping ===")
for label, idx in label2id.items():
    count = len(train[train["simplified_label"] == label])
    print(f"  {idx} → {label}: {count} samples")

✅ Labels encoded!

=== Label Mapping ===
  0 → Factual: 3638 samples
  1 → Misleading: 4488 samples
  2 → Uncertain: 2114 samples


In [5]:
# Cell 4 — Build PyTorch Dataset
from torch.utils.data import Dataset
from transformers import AutoTokenizer

# Load RoBERTa tokenizer
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
print("✅ Tokenizer loaded!")

class InfoTraceDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = str(self.data.iloc[idx]["clean_statement"])
        label = int(self.data.iloc[idx]["label_id"])

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label":          torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = InfoTraceDataset(train, tokenizer)
valid_dataset = InfoTraceDataset(valid, tokenizer)
test_dataset  = InfoTraceDataset(test, tokenizer)

print(f"✅ Datasets created!")
print(f"Train: {len(train_dataset)} samples")
print(f"Valid: {len(valid_dataset)} samples")
print(f"Test:  {len(test_dataset)} samples")

# Test one sample
sample = train_dataset[0]
print(f"\n=== Sample Tensor Shapes ===")
print(f"input_ids:      {sample['input_ids'].shape}")
print(f"attention_mask: {sample['attention_mask'].shape}")
print(f"label:          {sample['label']}")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded!
✅ Datasets created!
Train: 10240 samples
Valid: 1284 samples
Test:  1267 samples

=== Sample Tensor Shapes ===
input_ids:      torch.Size([128])
attention_mask: torch.Size([128])
label:          1


In [6]:
# Cell 5 — Create DataLoaders and Load Model
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print("✅ DataLoaders created!")
print(f"Train batches: {len(train_loader)}")
print(f"Valid batches: {len(valid_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Load RoBERTa model
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

model = model.to(device)

print("\n✅ RoBERTa model loaded!")
print(f"🖥️  Model is on: {device}")
print(f"📊 Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

✅ DataLoaders created!
Train batches: 640
Valid batches: 81
Test batches:  80


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



✅ RoBERTa model loaded!
🖥️  Model is on: cpu
📊 Number of parameters: 124,647,939


In [7]:
# Cell 6 — Setup Training
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Training settings
EPOCHS = 3
LEARNING_RATE = 2e-5

# Optimizer
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Learning rate scheduler
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

# Loss function
loss_fn = torch.nn.CrossEntropyLoss()

print("✅ Training setup complete!")
print(f"📊 Total training steps: {total_steps}")
print(f"🔥 Epochs: {EPOCHS}")
print(f"📈 Learning rate: {LEARNING_RATE}")
print(f"🔄 Warmup steps: {total_steps // 10}")

✅ Training setup complete!
📊 Total training steps: 1920
🔥 Epochs: 3
📈 Learning rate: 2e-05
🔄 Warmup steps: 192


In [ ]:
# Cell 7 — Train the Model
from tqdm import tqdm

def train_epoch(model, loader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    loop = tqdm(loader, desc="Training")
    for batch in loop:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        loop.set_postfix(loss=loss.item())

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        loop = tqdm(loader, desc="Evaluating")
        for batch in loop:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total


# Training loop
print("🚀 Starting Training...\n")
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(EPOCHS):
    print(f"{'='*50}")
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"{'='*50}")

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, device)
    val_loss, val_acc     = eval_epoch(model, valid_loader, loss_fn, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"\n📊 Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"📊 Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}\n")

print("✅ Training Complete!")

🚀 Starting Training...

Epoch 1/3


Evaluating: 100%|██████████| 81/81 [02:44<00:00,  2.04s/it]



📊 Train Loss: 1.0437 | Train Acc: 0.4792
📊 Val Loss:   0.9794 | Val Acc:   0.5428

Epoch 2/3


Evaluating: 100%|██████████| 81/81 [02:32<00:00,  1.88s/it]



📊 Train Loss: 0.9792 | Train Acc: 0.5466
📊 Val Loss:   0.9530 | Val Acc:   0.5522

Epoch 3/3


Evaluating: 100%|██████████| 81/81 [02:30<00:00,  1.86s/it]


📊 Train Loss: 0.8761 | Train Acc: 0.6053
📊 Val Loss:   0.9839 | Val Acc:   0.5405

✅ Training Complete!


: 

In [1]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import os

os.makedirs("../models", exist_ok=True)

# Save model and tokenizer
model.save_pretrained("../models/infotrace-roberta")
tokenizer.save_pretrained("../models/infotrace-roberta")

print("✅ Model saved to models/infotrace-roberta!")

NameError: name 'model' is not defined